In [0]:
from ingesta.motor import MotorIngesta
print("OK")

In [0]:
def read_config():

  # reads the client configuration from client.properties
  # and returns it as a key-value map
  config = {}

  with open("/Volumes/mastermsg001dbr/default/landing/client.properties") as fh:

    for line in fh:

      line = line.strip()

      if len(line) != 0 and line[0] != "#":

        parameter, value = line.strip().split('=', 1)
        config[parameter] = value.strip()

  return config

conf = read_config()

In [0]:
%pip install confluent-kafka

In [0]:
from confluent_kafka.admin import AdminClient
admin_client = AdminClient(conf)
topic_list = admin_client.list_topics().topics
print("Topics en el cluster: ")
topic_list

In [0]:
import json
from confluent_kafka import Producer

producer = Producer(conf)

mensaje = {
    "id": 3,
    "cliente_id": 300,
    "evento": "login",
    "timestamp":"2026-09-09T11:00:00"   
}
value = json.dumps(mensaje)

producer.produce("eventos_clientes", key="k2", value=value)
producer.flush()
print(f"Mensaje enviado a eventos_clientes: {value}")

In [0]:
config_path = '/Workspace/Users/masier02@ucm.es/farmia-ingesta/config/ingestion_config.json'
motor = MotorIngesta(spark, config_path)
queries = motor.ejecutar_streaming(available_now=True)
#queries = motor.ejecutar_streaming()

#for q in queries:
#  q.awaitTermination()

In [0]:
display(
    spark.read
    .format("delta")
    .load("/Volumes/mastermsg001dbr/default/bronze/eventos_clientes")
)

spark.read
.format("delta")
.load("/Volumes/mastermsg001dbr/default/bronze/eventos_clientes")
.printSchema()

In [0]:
from ingesta.config import load_config
from ingesta.streaming import write_streaming, read_streaming

config = load_config("config/ingestion_config.json")

eventos_config = next(
    dataset for dataset in config["streaming"]
    if dataset["dataset"] == "eventos_clientes"
)

df_eventos = read_streaming(spark, eventos_config)

query_eventos = write_streaming(
    eventos_config,
    df_eventos,
    available_now=True
)

query_eventos.awaitTermination()

In [0]:
from pyspark.sql import SparkSession
from ingesta.streaming import read_streaming

spark = SparkSession.builder.getOrCreate()

config = {
    "source": {
        "format": "kafka",
        "kafka_properties_file": "config/client.properties",
        "options": {
            "subscribe": "eventos_clientes",
            "startingOffsets": "earliest"
        },
        "key_format": "string",
        "key_subject": "eventos_clientes-key",
        "value_format": "json",
        "value_subject": "eventos_clientes-value",
        "json_schema": "id long, cliente_id long, evento string, timestamp string"
    }
}

df = read_streaming(spark, config)

df.printSchema()

In [0]:
from pyspark.sql import SparkSession
from ingesta.streaming import read_streaming, write_streaming

spark = SparkSession.builder.getOrCreate()

streaming_config = {
    "source": {
        "format": "kafka",
        "kafka_properties_file": "config/client.properties",
        "options": {
            "subscribe": "eventos_clientes",
            "startingOffsets": "earliest"
        },
        "key_format": "string",
        "key_subject": "eventos_clientes-key",
        "value_format": "json",
        "value_subject": "eventos_clientes-value",
        "json_schema": "id long, cliente_id long, evento string, timestamp string"
    },
    "sink": {
        "path": "/Volumes/mastermsg001dbr/default/bronze/eventos_clientes",
        "partition_columns": []
    }
}

df = read_streaming(spark, streaming_config)

query = write_streaming(
    streaming_config,
    df,
    available_now=True
)

query.awaitTermination()

print("Streaming terminado")

In [0]:
df.select(
    "key",
    "value.*",
    "topic",
    "partition",
    "offset",
    "_ingested_at"
).display()

In [0]:
from confluent_kafka import Producer
import json
import re

# Leer client.properties
properties = {}

with open("config/client.properties", "r") as f:
    for line in f:
        line = line.strip()

        if line and not line.startswith("#"):
            key, value = line.split("=", 1)
            properties[key] = value

# Extraer usuario y contraseña de kafka.sasl.jaas.config
jaas = properties["kafka.sasl.jaas.config"]

username = re.search(r"username='([^']+)'", jaas).group(1)
password = re.search(r"password='([^']+)'", jaas).group(1)

# Configuración para confluent-kafka
producer_config = {
    "bootstrap.servers": properties["kafka.bootstrap.servers"],
    "security.protocol": properties["kafka.security.protocol"],
    "sasl.mechanism": properties["kafka.sasl.mechanism"],
    "sasl.username": username,
    "sasl.password": password
}

producer = Producer(producer_config)

mensajes = [
    {"sensor_id": 1, "temperatura": 24.5, "humedad": 58.0},
    {"sensor_id": 2, "temperatura": 26.1, "humedad": 54.5},
    {"sensor_id": 3, "temperatura": 22.8, "humedad": 67.2}
]

for mensaje in mensajes:
    producer.produce(
        topic="sensores_iot",
        key=str(mensaje["sensor_id"]),
        value=json.dumps(mensaje)
    )

producer.flush()

print("3 mensajes enviados a sensores_iot")

In [0]:
from pyspark.sql import SparkSession
from ingesta.streaming import read_streaming, write_streaming

spark = SparkSession.builder.getOrCreate()

streaming_config = {
    "source": {
        "format": "kafka",
        "kafka_properties_file": "config/client.properties",
        "options": {
            "subscribePattern": "sensores_.*",
            "startingOffsets": "earliest"
        },
        "key_format": "string",
        "key_subject": "sensores_iot-key",
        "value_format": "json",
        "value_subject": "sensores_iot-value",
        "json_schema": "sensor_id long, temperatura double, humedad double"
    },
    "sink": {
        "path": "/Volumes/mastermsg001dbr/default/bronze/sensores_iot",
        "partition_columns": []
    }
}

df_sensores = read_streaming(spark, streaming_config)

query_sensores = write_streaming(
    streaming_config,
    df_sensores,
    available_now=True
)

query_sensores.awaitTermination()

print("Streaming sensores_iot terminado")

In [0]:
spark.read.format("delta").load(
    "/Volumes/mastermsg001dbr/default/bronze/sensores_iot"
).select(
    "key",
    "value.*",
    "topic",
    "partition",
    "offset",
    "_ingested_at"
).display()

In [0]:
from pyspark.sql import SparkSession
from ingesta.motor import MotorIngesta

spark = SparkSession.builder.getOrCreate()

motor = MotorIngesta(
    spark,
    "config/ingestion_config.json"
)

queries, errores = motor.ejecutar_streaming(available_now=True)

print("Queries:", len(queries))
print("Errores:", errores)

In [0]:
print("eventos_clientes:")
spark.read.format("delta").load(
    "/Volumes/mastermsg001dbr/default/bronze/eventos_clientes"
).count()

print("sensores_iot:")
spark.read.format("delta").load(
    "/Volumes/mastermsg001dbr/default/bronze/sensores_iot"
).count()

In [0]:
motor = MotorIngesta(
    spark,
    "config/ingestion_config.json"
)

print("Motor creado correctamente")

In [0]:
%pip install confluent-kafka[avro]

In [0]:
dbutils.library.restartPython()

In [0]:
from confluent_kafka import SerializingProducer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import StringSerializer

sr_conf = {
    "url": "https://psrc-zgxgq7p.us-east1.gcp.confluent.cloud",
    "basic.auth.user.info": "IIBN7RCM3BQ4AIVW:cfltPFXJcVac07dB3flxl+h3p1NRyD9Pq3NiOET9r1FL6fZiAnE75cXIlznHnj8g"
}

schema_registry_client = SchemaRegistryClient(sr_conf)

schema = """
{
  "type": "record",
  "name": "EventoCliente",
  "fields": [
    {"name": "id", "type": "long"},
    {"name": "cliente_id", "type": "long"},
    {"name": "evento", "type": "string"},
    {"name": "timestamp", "type": "string"}
  ]
}
"""

avro_serializer = AvroSerializer(
    schema_registry_client,
    schema
)

producer_conf = {
    "bootstrap.servers": "pkc-619z3.us-east1.gcp.confluent.cloud:9092",
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    "sasl.username": "4MFDPRDFINCDFBQZ",
    "sasl.password": "cfltC3Nq/hGnG7rolufdJiaaNhitmxSQlD05QPDDEZOZYLPaqFF/EIvVtM0a3q9w",
    "key.serializer": StringSerializer("utf_8"),
    "value.serializer": avro_serializer
}

producer = SerializingProducer(producer_conf)

producer.produce(
    topic="eventos_clientes",
    key="avro-test-1",
    value={
        "id": 100,
        "cliente_id": 25,
        "evento": "compra",
        "timestamp": "2026-09-10T11:30:00"
    }
)

producer.flush()

In [0]:

display(
    spark.read.format("delta").load(
        "/Volumes/mastermsg001dbr/default/bronze/sensores_iot"
    )
)

In [0]:
%pip install pytest

In [0]:
%env PYTHONDONTWRITEBYTECODE=1
!pytest -q -p no:cacheprovider